<a href="https://colab.research.google.com/github/DavidRR95/Fina/blob/main/Copia_de_Analisis_de_Portfolio.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# MBA 2025 - Indices Accionarios - Riesgo y Retorno

Profesor Fernando Díaz H.

In [ ]:
install.packages("tidyquant")

In [ ]:
library(tidyquant) # To download the data
library(timetk) # To manipulate the data series
library(tidyr)
library(ggplot2)
library(forcats)
library(dplyr)

# I Extraer Datos Bursátiles

## Data

Vamos a extraer información bursátil de las siguientes acciones:

* Apple Inc (AAPL);
* Wallmart (WMT);
* Intel (INTC)
* Exxon (XOM)
* L-M (LMT)


In [ ]:

tick <- c('AAPL', 'WMT', 'INTC', 'XOM', 'LMT')

In [ ]:
tick

 El paquete **tidyquant** es una herramienta para trabajar con datos financieros y análisis cuantitativos en el entorno **tidyverse**.

 La función **tq_get** se utiliza para extraer datos financieros directamente al entorno de R. Vamos a estraer los datos de las acciones para el período de 5 años que terminó ayer y las vamos a guardar en el objeto *price_data*:

In [ ]:
price_data <- tq_get(tick,
                     from = '2020-01-01',
                     to = '2024-12-31',
                     get = 'stock.prices')

In [ ]:
class(price_data)
price_data

## Retornos Diarios

A continuación, calcularemos la rentabilidad diaria de estos valores. Utilizaremos los rendimientos logarítmicos y guardaremos la información en el objeto *log_red_tidy*:


In [ ]:
log_ret_tidy <- price_data %>%
  group_by(symbol) %>%
  tq_transmute(select = adjusted,
               mutate_fun = periodReturn,
               period = 'daily',
               col_rename = 'ret',
               type = 'log')

In [ ]:
log_ret_tidy

El operador %>% (**pipe operator**) es una característica poderosa del paquete **dplyr** que  permite encadenar secuencialmente varias operaciones en un flujo de trabajo de análisis de datos. Esto hace que el código sea más legible, ya que las operaciones se ejecutan de izquierda a derecha, pasando el resultado de la operación anterior como el primer argumento de la siguiente operación.

La función **group_by()** se usa para agrupar los datos en un marco de datos según una o varias variables.

 La función **tq_transmute** se utiliza para transformar y resumir datos financieros



In [ ]:
head(log_ret_tidy)

## Formato "Ancho"

La función **pivot_wider** es parte del paquete **tidyr** en R y se utiliza para transformar datos de formato largo a formato ancho, creando nuevas columnas basadas en valores únicos en una columna y llenando esas columnas con valores de otra columna.

In [ ]:
log_ret_xts <- log_ret_tidy %>%
  pivot_wider(names_from = symbol, values_from =ret) %>%
  tk_xts()

head(log_ret_xts)

Ahora sí tenemos los datos como los necesitamos.

# II Estadísticos Requeridos

## Rentabilidad media diaria de cada activo.

In [ ]:
mean_ret <- colMeans(log_ret_xts)
print(round(mean_ret, 5))

## Varianzas y Covarianzas

Calcularemos la matriz de covarianzas de todas estas acciones y las anualizaremos multiplicándola por 252.

In [ ]:
cov_mat <- cov(log_ret_xts)*252

print(round(cov_mat,4))

# III Single Portfolio

Para calcular la rentabilidad de la cartera y el riesgo (desviación típica) necesitaremos:

* Rentabilidad media de los activos
* Ponderaciones del portfolio
* Matriz de covarianza de todos los activos


## Ponderaciones Aleatorias

La función **runif** es una función incorporada en R que se utiliza para generar números aleatorios según una distribución uniforme.

In [ ]:
wts <- runif(n = length(tick), min = 0, max = 1)
wts <- wts/sum(wts)
print(wts)
print(sum(wts))

## Retornos y Riesgo Anualizados


$E\left[\tilde{r}_{p}\right]=\sum_{i=1}^{N}w_{i}E\left[\tilde{r}_{i}\right]$

$Var\left[\tilde{r}_{p}\right]=\sum_{i=1}^{N}w_{i}^{2}\sigma_{ii}+\underset{i\neq j}{\sum_{i=1}^{N}\sum_{j=1}^{N}w_{i}w_{j}\sigma_{ij}}=\sum_{i=1}^{N}\sum_{j=1}^{N}w_{i}w_{j}\sigma_{ij}$


$
E\left[\tilde{r}_{p}\right]=\left[E\left[\tilde{r}_{1}\right]\,E\left[\tilde{r}_{2}\right]\,...\,E\left[\tilde{r}_{N}\right]\right]\left[\begin{array}{c}
w_{1}\\
w_{2}\\
\vdots\\
w_{N}
\end{array}\right]=R'W
$
$
Var\left[\tilde{r}_{p}\right]=\left[w_{1}\,w_{2}\,...\,w_{N}\right]\left[\begin{array}{cccc}
\sigma_{11} & \sigma_{12} & \cdots & \sigma_{1N}\\
\sigma_{21} & \sigma_{22} & \cdots & \sigma_{2N}\\
\vdots & \vdots & \ddots & \vdots\\
\sigma_{N1} & \sigma_{N2} & \cdots & \sigma_{NN}
\end{array}\right]\left[\begin{array}{c}
w_{1}\\
w_{2}\\
\vdots\\
w_{N}
\end{array}\right]=W'\Sigma W
$




In [ ]:
port_returns <- (sum(wts * mean_ret) + 1)^252 - 1
print(port_returns)

port_risk <- sqrt(t(wts) %*% (cov_mat %*% wts))
print(port_risk)

## Sharpe Ratio

Necesitamos la tasa libre de riesgo en USA: http://www.worldgovernmentbonds.com/

In [ ]:
rf <- 0.04413

sharpe_ratio <- (port_returns-rf)/port_risk
print(sharpe_ratio)

# IV Randomizando sobre N Portfolios

OK. Primero debemos definir cuántos portfolios queremos en el gráfico. Vamos a crear también el objeto *all_wts* para almacenar los datos:

In [ ]:
num_port <- 50000
all_wts <- matrix(nrow = num_port,
                  ncol = length(tick))

Tenemos que:
* Crear  un vector vacío para almacenar los retornos de cada cartera - **port_returns**
* Crear un vector vacío para almacenar la  desviación estándar de cada cartera - **port_risk**
* Crear un vector vacío para almacenar el  ratio de Sharpe de cada cartera - **sharpe_ratio**

In [ ]:
port_returns <- vector('numeric', length = num_port)
port_risk <- vector('numeric', length = num_port)
sharpe_ratio <- vector('numeric', length = num_port)


Ahora, con un comando *for*, vamos a ir llenando cada uno de estos vectores con los resultados de repetir (III) tantas veces como portfolios queremos (**num_port**). Para esto debemos **indexar** las matrices o vectores.

Por ejemplo, para acceder al elemento en la segunda fila y tercera columna escribimos *mat[2,3]*.

Para acceder a la segunda fila completa, escribimos *mat[2, ]*.

In [ ]:
for (i in seq_along(port_returns)) {

  wts <- runif(length(tick))
  wts <- wts/sum(wts)

  # Storing weight in the matrix
  all_wts[i,] <- wts

  # Portfolio returns

  port_ret <- sum(wts * mean_ret)
  port_ret <- ((port_ret + 1)^252) - 1

  # Storing Portfolio Returns values
  port_returns[i] <- port_ret


  # Creating and storing portfolio risk
  port_sd <- sqrt(t(wts) %*% (cov_mat  %*% wts))
  port_risk[i] <- port_sd

  # Creating and storing Portfolio Sharpe Ratios
  # Assuming 0% Risk free rate

  sr <- (port_ret-rf)/port_sd
  sharpe_ratio[i] <- sr

}

Y guardamos los resultados en una tabla (**tibble**):

In [ ]:
portfolio_values <- tibble(Return = port_returns,
                           Risk = port_risk,
                           SharpeRatio = sharpe_ratio)

In [ ]:
head(portfolio_values)

Ahora queremos pegar los pesos de cada portfolio a la tabla que acabamos de crear.

In [ ]:
all_wts <- tk_tbl(all_wts)

In [ ]:
head(all_wts)
class(all_wts)

Las columnas corresponden a los pesos en cada una de las acciones. Pongámosle nombre:

In [ ]:
colnames(all_wts) <- colnames(log_ret_xts)
head(all_wts)

Finalmente, juntamos ambas tablas con el comando **cbind**

In [ ]:
portfolio_values <- cbind(all_wts, portfolio_values)
head(portfolio_values)

¡Esto es justamente lo que queríamos!

# V Gráficos

## El Portfolio de Mínima Varianza

In [ ]:
min_var <- portfolio_values[which.min(portfolio_values$Risk),]
min_var

Ahora graficamos usando **ggplot**

In [ ]:

p <- min_var %>%
  gather(all_of(tick), key = Asset,
         value = Weights) %>%
  mutate(Asset = as.factor(Asset)) %>%
  ggplot(aes(x = fct_reorder(Asset,Weights), y = Weights, fill = Asset)) +
  geom_bar(stat = 'identity') +
  theme_minimal() +
  labs(x = 'Assets', y = 'Weights', title = "Minimum Variance Portfolio Weights") +
  scale_y_continuous(labels = scales::percent)

p

## El Portfolio de Tangencia

In [ ]:
max_sr <- portfolio_values[which.max(portfolio_values$SharpeRatio),]
max_sr

In [ ]:
p <- max_sr %>%
  gather(all_of(tick), key = Asset,
         value = Weights) %>%
  mutate(Asset = as.factor(Asset)) %>%
  ggplot(aes(x = fct_reorder(Asset,Weights), y = Weights, fill = Asset)) +
  geom_bar(stat = 'identity') +
  theme_minimal() +
  labs(x = 'Assets', y = 'Weights', title = "Tangency Portfolio Weights") +
  scale_y_continuous(labels = scales::percent)
p

## La Frontera Eficiente

In [ ]:

p <- portfolio_values %>%
  ggplot(aes(x = Risk, y = Return, color = SharpeRatio)) +
  geom_point() +
  theme_classic() +
  scale_y_continuous(labels = scales::percent) +
  scale_x_continuous(labels = scales::percent) +
  labs(x = 'Annualized Risk',
       y = 'Annualized Returns',
       title = "Portfolio Optimization & Efficient Frontier") +
  geom_point(aes(x = Risk,
                 y = Return), data = min_var, color = 'black',size=4) +
  geom_point(aes(x = Risk,
                 y = Return), data = max_sr, color = 'red',size=4) +
  annotate('text', x = 0.20, y = 0.285, label = "Tangency Portfolio",
           color = 'red') +
  annotate('text', x = 0.18, y = 0.10, label = "Minimum Variance Portfolio",
           color = 'black')
p

In [ ]:
p <- portfolio_values %>%
  ggplot(aes(x = Risk, y = Return, color = SharpeRatio)) +
  geom_point() +
  theme_classic() +
  scale_y_continuous(labels = scales::percent) +
  scale_x_continuous(labels = scales::percent) +
  labs(x = 'Annualized Risk',
       y = 'Annualized Returns',
       title = "Portfolio Optimization & Efficient Frontier") +
  geom_point(aes(x = Risk,
                 y = Return), data = min_var, color = 'black', size = 4) +
  geom_point(aes(x = Risk,
                 y = Return), data = max_sr, color = 'red', size = 4) +
  geom_label(aes(x = 0.20, y = 0.285, label = "Tangency Portfolio"),
             color = 'red', fill = 'white', label.size = 0.25) +
  geom_label(aes(x = 0.18, y = 0.10, label = "Minimum Variance Portfolio"),
             color = 'black', fill = 'white', label.size = 0.25)


p


Y si lo queremos grabar:

In [ ]:
ggsave(p, file="Frontera_Ef.png", device=cairo_ps,width=10, height=5)